In [2]:
import os

import numpy as np
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision.transforms import transforms

from PIL import Image

In [3]:
DATASET_PATH = r"/itf-fi-ml/shared/courses/IN3310/mandatory1_data"

In [4]:
# List all class names (folder names) and sort list of names
class_names = os.listdir(DATASET_PATH)
class_names.sort()
class_names

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

In [5]:
# Create nested list containing lists of filenames from each class, where the first index (list) refer to 
# the same class as in the (sorted) class_names list
class_filenames = []

for class_name in class_names:
    class_files_path = os.path.join(DATASET_PATH, class_name)
    class_filenames.append(os.listdir(class_files_path))

In [ ]:
class_filenames

In [7]:
# Sum number of elements contained in each list of filenames per class, store in a np.array with index referring to same index as in class_names
files_per_class = np.zeros(len(class_filenames))

for class_number, filenames in enumerate(class_filenames):
    files_per_class[class_number] = len(filenames)
    print(f"class: {class_number}, {class_names[class_number]}: {len(filenames)} files")
    

print(f"Files per class: {files_per_class}")

files_total = np.sum(files_per_class)
files_relative = np.copy(files_per_class)/files_total
print(f"Files total: {files_total}")
print(f"Realtive number of files per class {files_relative}")

class: 0, buildings: 2628 files
class: 1, forest: 2745 files
class: 2, glacier: 2957 files
class: 3, mountain: 3037 files
class: 4, sea: 2784 files
class: 5, street: 2883 files
Files per class: [2628. 2745. 2957. 3037. 2784. 2883.]
Files total: 17034.0
Realtive number of files per class [0.15427968 0.16114829 0.17359399 0.17829048 0.16343783 0.16924974]


In [ ]:
# Create array of indices for each file in each of the classes, format: [[class_number, index], ... ]
class_filenames_indices = []

for class_number, class_filenames_list in enumerate(class_filenames):
    idx_class_filenames = list(range(len(class_filenames_list)))

    for idx in idx_class_filenames:
        class_filenames_indices.append([class_number, idx])

class_filenames_indices = np.asarray(class_filenames_indices, dtype=np.int32)
class_filenames_indices

In [9]:
# Make array of targets (classes) y to correspond to list of [class_number, file_index] pairs
x = class_filenames_indices
y = class_filenames_indices[:,0]

# Two step process to split the dataset into train-, val- and test-set, about 70, 10 and 20% respectively
x_train, x_valtest, y_train, y_valtest = train_test_split(x, y, train_size=0.70, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_valtest, y_valtest, train_size=0.40, test_size=0.60, stratify=y_valtest)

In [10]:
# Turn numpy array of arrays into tuple of tuples to be used with .isdisjoint 
set_x_train = set(tuple(map(tuple, x_train.tolist())))
set_x_val = set(tuple(map(tuple, x_val.tolist())))
set_x_test = set(tuple(map(tuple, x_test.tolist())))

print("Disjoint:")
print(set_x_train.isdisjoint(set_x_val))
print(set_x_train.isdisjoint(set_x_test))
print(set_x_val.isdisjoint(set_x_test))

print("--------------")

print("Same length:")
print(len(set_x_train) == x_train.shape[0])
print(len(set_x_val) == x_val.shape[0])
print(len(set_x_test) == x_test.shape[0])

Disjoint:
True
True
True
--------------
Same length:
True
True
True


In [ ]:
# Extract the file paths for each file in each of the datasets (train, val, and test) and store in list 

x_train_file_paths = []

for class_idx, filename_idx in x_train:
    print(class_idx, filename_idx, class_filenames[class_idx][filename_idx])

    class_name = class_names[class_idx]
    filename = class_filenames[class_idx][filename_idx]
    file_path = os.path.join(DATASET_PATH, class_name, filename)
    x_train_file_paths.append(file_path)
    print(file_path)

x_val_file_paths = []

for class_idx, filename_idx in x_val:
    print(class_idx, filename_idx, class_filenames[class_idx][filename_idx])

    class_name = class_names[class_idx]
    filename = class_filenames[class_idx][filename_idx]
    file_path = os.path.join(DATASET_PATH, class_name, filename)
    x_val_file_paths.append(file_path)
    print(file_path)

x_test_file_paths = []

for class_idx, filename_idx in x_test:
    print(class_idx, filename_idx, class_filenames[class_idx][filename_idx])

    class_name = class_names[class_idx]
    filename = class_filenames[class_idx][filename_idx]
    file_path = os.path.join(DATASET_PATH, class_name, filename)
    x_test_file_paths.append(file_path)
    print(file_path)

In [12]:
# Check that any filename does not appear in more than one of the three datasets (train, val, test)

print(set(x_train_file_paths).isdisjoint(set(x_val_file_paths)))
print(set(x_train_file_paths).isdisjoint(set(x_test_file_paths)))
print(set(x_val_file_paths).isdisjoint(set(x_test_file_paths)))

True
True
True


In [13]:
def find_class_names_filenames(img_dir):
    """
    Find class names based on folder names and list filenames class wise.
    
    Args:
        img_dir (str): root directory path, must folders of image files, each folder representing a class
    
    Return:

        class_names (str): list of class names, sorted
        class_filenames (list): list containing lists with filenames, each the indices of each sublist correspond to same index in class_names 
    """

    # List all class names (folder names) and sort the list
    class_names = os.listdir(img_dir)
    class_names.sort()

    # Create list containing lists of all filenames from each class
    # Each list containing filenames have same index as the index of class name in the sorted list class_names
    class_filenames = []
    for class_name in class_names:
        class_files_path = os.path.join(img_dir, class_name)
        class_filenames.append(os.listdir(class_files_path))

    return class_names, class_filenames

def get_path_from_dataset_indices(dataset, class_names, class_filenames, img_dir):
    """
    Helper function for stratified_split_data_paths().
    Input:  dataset --> list of [class_number, file_index] pairs
            class_names --> list of class names
            class_filenames --> list of lists, where each sub list contains filenames of class
            img_dir --> root directory with class
    
    Return: list of filepaths
    """
    dataset_file_paths = []
    for class_idx, filename_idx in dataset:
        # print(class_idx, filename_idx, class_filenames[class_idx][filename_idx])

        class_name = class_names[class_idx]
        filename = class_filenames[class_idx][filename_idx]
        file_path = os.path.join(img_dir, class_name, filename)
        dataset_file_paths.append(file_path)

    return dataset_file_paths

def stratified_split_data_paths(img_dir, class_names, class_filenames):
    """
    1. Create array of indices for each file in each of the classes, in format: [[class_number, img_index], ... ]
    2. Use array of [class_number, img_index] pairs as x, and create a class_number target array, y 
    3. Split into into train, val and, test set
    4. Check if sets are disjoint
    
    Return: Lists of img paths and corresponding lists of class numbers
    """

    # Create one large array containing [class_number, filename_index] pairs of all the images based on the ordering in class_filenames
    class_filenames_indices = []
    for class_number, class_filenames_list in enumerate(class_filenames):
        idx_class_filenames = list(range(len(class_filenames_list)))

        for idx in idx_class_filenames:
            class_filenames_indices.append([class_number, idx])

    class_filenames_indices = np.asarray(class_filenames_indices, dtype=np.int32)

    # Make array of target (class number) y to correspond to list of [class_number, file_index] pairs
    x = class_filenames_indices
    y = class_filenames_indices[:,0]

    # Two step process of spliting the indices into train-, val- and test-set, about 70, 10 and 20% respectively
    x_train_indices, x_valtest_indices, y_train, y_valtest = train_test_split(x, y, train_size=0.70, stratify=y)
    x_val_indices, x_test_indices, y_val, y_test= train_test_split(x_valtest_indices, 
                                                                                    y_valtest, 
                                                                                    train_size=0.40, 
                                                                                    test_size=0.60, 
                                                                                    stratify=y_valtest)

    # Get the paths from the indices
    x_train_paths = get_path_from_dataset_indices(x_train_indices, class_names, class_filenames, img_dir)
    x_val_paths = get_path_from_dataset_indices(x_val_indices, class_names, class_filenames, img_dir)
    x_test_paths = get_path_from_dataset_indices(x_test_indices, class_names, class_filenames, img_dir)

    # Check if the sets are disjoint
    not_disjoint = 0
    if not set(x_train_paths).isdisjoint(set(x_val_paths)):
        print("Train and val set not disjoint")
        not_disjoint += 0

    if not set(x_train_paths).isdisjoint(set(x_test_paths)):
        print("Train and test set not disjoint")
        not_disjoint += 0

    if not set(x_val_paths).isdisjoint(set(x_test_paths)):
        print("Val and test set not disjoint")
        not_disjoint += 0

    if not_disjoint > 0:
        raise ValueError("Datasets not disjoint")


    return x_train_paths, x_val_paths, x_test_paths, y_train, y_val, y_test

In [14]:
# Class which turn file paths into pipeline-friendly object  
# Inherit abstract class: torch.utils.data.Dataset, this to make it compatible with the pytorch pipeline
# Need __init__, __len__ and __getitem__
# Inspired by 02_CNN_Example.ipynb

class NatureCityScenesDataset(torch.utils.data.Dataset):
    def __init__(self, x_paths, y_class_number, transform=None):

        self.image_paths = x_paths
        self.image_label = y_class_number

        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        # Convert to RGB (if some images are RGBA or Grayscale)
        image = Image.open(img_path).convert("RGB")

        label = int(self.image_label[idx])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [15]:
class_names, class_filenames = find_class_names_filenames(DATASET_PATH)
x_train_paths, x_val_paths, x_test_paths, y_train, y_val, y_test = stratified_split_data_paths(DATASET_PATH, class_names, class_filenames)

In [16]:
train_set = NatureCityScenesDataset(x_train_paths, y_train)
val_set = NatureCityScenesDataset(x_val_paths, y_val)
test_set = NatureCityScenesDataset(x_test_paths, y_test)

BATCH_SIZE = 32
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_load = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)